In [13]:
import os
os.chdir(r'c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks')
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks


# Step 5: Hybrid Recommendation Model (BM25 + Transformer)

In this final step, we implement a **Hybrid Scoring System**. 

### Why Hybrid?
- **BM25** is excellent at exact keyword matching, which is crucial when reviews have a very specific vocabulary (e.g., "pizza", "museum").
- **Transformers** (MPNet) excel at semantic similarity, understanding when words are related even if they don't match exactly (e.g., "cheap" vs "budget").

By combining both, we can leverage exact term overlap while using the neural model to "break ties" and provide deeper semantic mapping for sub-categories (Level 2).

In [14]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from scipy.stats import wilcoxon
import warnings
import sys, os

# Add parent directory to path to import utils
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from utils.evaluation import eval_level_1, extract_subcategories, eval_level_2

warnings.filterwarnings('ignore')

# 1. Load the prepared data
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)

# Filter relevant evaluation columns
eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')

# 2. Train/Test Split (Same 50/50 split and seed as previous models)
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Total places: {len(df_merged)}")
print(f"Train (Query) places: {len(train_df)}")
print(f"Test (Corpus) places: {len(test_df)}")

Total places: 1835
Train (Query) places: 917
Test (Corpus) places: 918


## 1. BM25 Scoring
We first compute the statistical similarity scores based on term frequency and document length.

In [15]:
corpus_tokens = [str(t).lower().split() for t in test_df['top_100_words'].fillna('').tolist()]
bm25 = BM25Okapi(corpus_tokens)

print("Computing BM25 scores for queries...")
bm25_scores = []
for doc in train_df['top_100_words'].fillna('').tolist():
    bm25_scores.append(bm25.get_scores(doc.lower().split()))
bm25_scores = np.array(bm25_scores)

Computing BM25 scores for queries...


## 2. Transformer Scoring (MPNET)
We use the more powerful `all-mpnet-base-v2` model to capture semantic embeddings.

In [16]:
print("Loading all-mpnet-base-v2 and encoding signatures...")
model = SentenceTransformer('all-mpnet-base-v2')

test_vectors = model.encode(test_df['top_100_words'].fillna('').tolist(), show_progress_bar=True)
train_vectors = model.encode(train_df['top_100_words'].fillna('').tolist(), show_progress_bar=True)

st_scores = cosine_similarity(train_vectors, test_vectors)

Loading all-mpnet-base-v2 and encoding signatures...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

## 3. Hybrid Normalization & Weighting
Since BM25 scores are unbounded and Cosine Similarity is between -1 and 1, we apply Min-Max scaling to combine them effectively.

In [17]:
# --- GRID SEARCH FOR OPTIMAL FUSION (ALPHA) ---
from utils.evaluation import per_row_minmax

# Normalize scores per-query (FIX: avoid dynamic range compression)
bm25_norm = per_row_minmax(bm25_scores)
st_norm = per_row_minmax(st_scores)

def perform_fusion(alpha):
    # Higher alpha = more BM25 dominance
    fusion_scores = (alpha * bm25_norm) + ((1 - alpha) * st_norm)
    hybrid_ranked_indices = [np.argsort(fusion_scores[i])[::-1] for i in range(len(train_df))]
    return np.array(hybrid_ranked_indices)

# Helper for quick evaluation for Grid Search
def quick_eval(ranked_indices_matrix):
    l2_errors = []
    for i in range(len(train_df)):
        row = train_df.iloc[i]
        q_subcats = extract_subcategories(row)
        if q_subcats:
            err2 = eval_level_2(q_subcats, ranked_indices_matrix[i], test_subcats_list)
            if err2 is not None:
                l2_errors.append(err2)
    return np.mean(l2_errors)

best_alpha = 1.0
min_l2_err = float('inf')
results = {}

print("Grid search for Alpha (BM25 weight)...")
for alpha in [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 1.0]:
    reranked_indices = perform_fusion(alpha)
    l2_avg = quick_eval(reranked_indices)
    results[alpha] = l2_avg
    print(f"Alpha={alpha:.2f} -> Level 2 Error: {l2_avg:.2f}")
    if l2_avg < min_l2_err:
        min_l2_err = l2_avg
        best_alpha = alpha

print(f"\nOptimal Alpha found: {best_alpha} with Level 2 Error: {min_l2_err:.2f}")

# Final hybrid ranked indices for Cell 10
hybrid_ranked_indices = perform_fusion(best_alpha)

Grid search for Alpha (BM25 weight)...
Alpha=0.00 -> Level 2 Error: 8.69
Alpha=0.20 -> Level 2 Error: 5.88


Alpha=0.40 -> Level 2 Error: 4.09
Alpha=0.60 -> Level 2 Error: 3.88
Alpha=0.80 -> Level 2 Error: 4.11
Alpha=0.90 -> Level 2 Error: 4.25


Alpha=0.95 -> Level 2 Error: 4.33
Alpha=1.00 -> Level 2 Error: 4.41

Optimal Alpha found: 0.6 with Level 2 Error: 3.88


## 4. Final Evaluation
We evaluate the hybrid model against the Baseline BM25 scores.

In [18]:
# --- FINAL EVALUATION & STATISTICAL SIGNIFICANCE ---

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]
test_types = test_df['typeR'].values

def run_full_eval(ranked_indices_matrix):
    l1_errors, l2_errors = [], []
    for i in range(len(train_df)):
        row = train_df.iloc[i]
        ranked_idx = ranked_indices_matrix[i]
        
        # Level 1
        test_typeR_list = test_types[ranked_idx].tolist()
        err1 = eval_level_1(row['typeR'], test_typeR_list)
        l1_errors.append(err1)
        
        # Level 2
        q_subcats = extract_subcategories(row)
        err2 = eval_level_2(q_subcats, ranked_idx, test_subcats_list)
        l2_errors.append(err2)
        
    return l1_errors, l2_errors

# Get per-query error vectors for BM25 and Hybrid
bm25_ranked_idx = [np.argsort(bm25_scores[i])[::-1] for i in range(len(train_df))]
bm25_l1_errs, bm25_l2_errs = run_full_eval(bm25_ranked_idx)
hyb_l1_errs, hyb_l2_errs = run_full_eval(hybrid_ranked_indices)

# Filter out None values (undefined cases) for reporting means
bm25_l1_valid = np.array([e for e in bm25_l1_errs if e is not None], dtype=float)
bm25_l2_valid = np.array([e for e in bm25_l2_errs if e is not None], dtype=float)
hyb_l1_valid = np.array([e for e in hyb_l1_errs if e is not None], dtype=float)
hyb_l2_valid = np.array([e for e in hyb_l2_errs if e is not None], dtype=float)

print("-" * 40)
print(f"BM25 Baseline      - L1 Error Mean: {np.mean(bm25_l1_valid):.2f} | L2 Error Mean: {np.mean(bm25_l2_valid):.2f}")
print(f"Hybrid (Re-Rank)   - L1 Error Mean: {np.mean(hyb_l1_valid):.2f} | L2 Error Mean: {np.mean(hyb_l2_valid):.2f}")
print("-" * 40)

# STATISTICAL SIGNIFICANCE (Wilcoxon signed-rank test on per-query diffs)
# Level 2 is the discriminative target for re-ranking
common_l2_indices = [i for i in range(len(bm25_l2_errs)) if bm25_l2_errs[i] is not None and hyb_l2_errs[i] is not None]
l2_bm25_vector = np.array([bm25_l2_errs[i] for i in common_l2_indices], dtype=float)
l2_hyb_vector = np.array([hyb_l2_errs[i] for i in common_l2_indices], dtype=float)

# Wilcoxon requires differences to be non-zero to be meaningful, but the function handles ties.
# However, if all differences are zero, it will fail.
if not np.array_equal(l2_bm25_vector, l2_hyb_vector):
    # Wilcoxon test requires at least one non-zero difference
    diff = l2_bm25_vector - l2_hyb_vector
    if np.any(diff != 0):
        stat, p = wilcoxon(l2_bm25_vector, l2_hyb_vector)
        print(f"Wilcoxon Significance (Level 2): p-value = {p:.4f}")
        if p < 0.05:
            print("Model improvement is statistically significant (p < 0.05).")
        else:
            print("Model improvement is not statistically significant (p >= 0.05).")
    else:
        print("All differences are zero (identical performance on valid samples).")
else:
    print("Baseline and Hybrid model yielding identical Level 2 scores.")

if np.mean(hyb_l2_valid) < np.mean(bm25_l2_valid):
    print("\nSUCCESS: The Hybrid (Re-rank) model achieved a lower Ranking Error than the baseline!")
else:
    print("\nNOTE: The baseline remains superior. Consider tuning K or alpha.")

----------------------------------------
BM25 Baseline      - L1 Error Mean: 0.64 | L2 Error Mean: 4.41
Hybrid (Re-Rank)   - L1 Error Mean: 0.70 | L2 Error Mean: 3.88
----------------------------------------
Wilcoxon Significance (Level 2): p-value = 0.1909
Model improvement is not statistically significant (p >= 0.05).

SUCCESS: The Hybrid (Re-rank) model achieved a lower Ranking Error than the baseline!
